In [ ]:
!pip install MEDS-Inspect

In [ ]:
MEDS_Inspect_cache "/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"

In [ ]:
!MEDS_Inspect port=8052 +initial_path="/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort"

# Imports

In [ ]:
import os
import pandas as pd
import subprocess
import numpy as np
import hail as hl
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from datetime import datetime

In [ ]:
import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))
os.environ["WORKSPACE_CDR"] = "wb-silky-artichoke-2408.C2025Q4R6"

In [ ]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 200g "
    "--conf spark.driver.maxResultSize=16g "
    "--conf spark.default.parallelism=64 "
    "--conf spark.sql.shuffle.partitions=256 "
    "pyspark-shell"
)

import hail as hl

hl.init(
    master="local[32]",
    idempotent=True,
    default_reference = "GRCh38"
)

In [ ]:
hl.stop()
hl.init(default_reference = "GRCh38")

# Clinical Data

In [ ]:
dataset_08947253_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id"""

dataset_08947253_person_df = pd.read_gbq(
    dataset_08947253_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_08947253_person_df.head(5)

In [ ]:
DATA_BUCKET = '/home/jupyter/workspace/data_bucket'
GENETIC_FOLDER = f'{DATA_BUCKET}/v9_gen_data'
dataset_08947253_person_df.to_csv(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv", sep = "\t", index=False)
dataset_hl = (hl.import_table(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv",
                              types={'person_id':hl.tstr},
                              impute=True,
                              key='person_id')
             )

In [ ]:
dataset_hl.summarize()

# Genetic Data

In [ ]:
import hail as hl
import pandas as pd

vat_path = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/aux/vat/vat_complete.bgz.tsv.gz"
)

vat_table = hl.import_table(
    vat_path,
    force=True,
    quote='"',
    delimiter="\t",
    force_bgz=True,
    types={
        "position": hl.tint32,
        "contig": hl.tstr,
        "ref_allele": hl.tstr,
        "alt_allele": hl.tstr,
    },
)

vat_table.describe()

In [ ]:
transcripts_literal = hl.literal(set(transcripts_of_interest))

# Remove transcript version suffixes:
# ENST00000372037.5 -> ENST00000372037
transcript_vat_table = vat_table.annotate(
    transcript_id=(
        hl.or_else(vat_table.transcript, "")
        .split(r"\.")[0]
    )
)

transcript_vat_table = transcript_vat_table.filter(
    transcripts_literal.contains(
        transcript_vat_table.transcript_id
    )
)

transcript_vat_table.describe()

In [ ]:
# A VAT row passes when at least one comma-separated classification
# is in this set.
accepted_classifications = hl.literal({
    "pathogenic",
    "likely pathogenic",
    "likely risk allele",
    "risk factor",
})

# Normalize classifications such as:
# "likely pathogenic,pathogenic"
# "likely pathogenic, pathogenic"
# into the same component representation.
filtered_vat_table = transcript_vat_table.annotate(
    _classification_terms=(
        hl.or_else(
            transcript_vat_table.clinvar_classification,
            ""
        )
        .lower()
        .split(",")
        .map(lambda value: value.strip())
    )
)

filtered_vat_table = filtered_vat_table.filter(
    hl.any(
        lambda classification:
            accepted_classifications.contains(classification),
        filtered_vat_table._classification_terms,
    )
)

# The temporary normalized field is no longer needed.
filtered_vat_table = filtered_vat_table.drop(
    "_classification_terms"
)

# Create the same row-key fields used by the MatrixTable.
filtered_vat_table = filtered_vat_table.annotate(
    locus=hl.locus(
        filtered_vat_table.contig,
        filtered_vat_table.position,
        reference_genome="GRCh38",
    ),
    alleles=[
        filtered_vat_table.ref_allele,
        filtered_vat_table.alt_allele,
    ],
)

# Exclude annotations that are only upstream or downstream.
excluded_consequences = hl.literal({
    "downstream_gene_variant",
    "upstream_gene_variant",
})

filtered_vat_table = filtered_vat_table.filter(
    hl.is_missing(filtered_vat_table.consequence)
    | ~excluded_consequences.contains(
        filtered_vat_table.consequence
    )
)

# Key individual VAT rows by variant.
filtered_vat_table = filtered_vat_table.key_by(
    "locus",
    "alleles",
)

# A variant can have multiple relevant transcript rows.
# Aggregate them into one uniquely keyed row per variant.
filtered_vat_by_variant = (
    filtered_vat_table
    .group_by(
        locus=filtered_vat_table.locus,
        alleles=filtered_vat_table.alleles,
    )
    .aggregate(
        # Preserve every matching VAT transcript annotation.
        annotations=hl.agg.collect(
            filtered_vat_table.row_value
        ),

        gene_symbols=hl.agg.filter(
            hl.is_defined(filtered_vat_table.gene_symbol)
            & (filtered_vat_table.gene_symbol != ""),
            hl.agg.collect_as_set(
                filtered_vat_table.gene_symbol
            ),
        ),

        transcripts=hl.agg.filter(
            hl.is_defined(filtered_vat_table.transcript)
            & (filtered_vat_table.transcript != ""),
            hl.agg.collect_as_set(
                filtered_vat_table.transcript
            ),
        ),

        consequences=hl.agg.filter(
            hl.is_defined(filtered_vat_table.consequence)
            & (filtered_vat_table.consequence != ""),
            hl.agg.collect_as_set(
                filtered_vat_table.consequence
            ),
        ),

        clinvar_classifications=hl.agg.filter(
            hl.is_defined(
                filtered_vat_table.clinvar_classification
            ),
            hl.agg.collect_as_set(
                filtered_vat_table.clinvar_classification
            ),
        ),
    )
)

filtered_vat_by_variant.describe()

In [ ]:
df = pd.read_csv(f"{GENETIC_FOLDER}/df_whole_DS_v9.tsv", sep = "\t")
df_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/df_whole_DS_v8.tsv', sep='t')

In [ ]:
meta_v9 = pd.read_csv(f"{GENETIC_FOLDER}/metadata_v9.csv")
meta_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/metadata_v8.csv')

In [ ]:
samples_v8 = set(meta_v8["s"].dropna().unique())
samples_v9 = set(meta_v9["s"].dropna().unique())


print("v8 total:", len(samples_v8))
print("v9 total:", len(samples_v9))
print("v8 also in v9:", len(samples_v8 & samples_v9))
print("v8 missing from v9:", len(samples_v8 - samples_v9))
print("new in v9:", len(samples_v9 - samples_v8))

In [ ]:
meta_v9[meta_v9['s']==int(list(samples_v8 & samples_v9 & samples_v8_filt & (samples_v8_filt - samples_v9_filt))[11])]

In [ ]:
meta_v8[meta_v8['s']==int(list(samples_v8 & samples_v9 & samples_v8_filt & (samples_v8_filt - samples_v9_filt))[11])]

In [ ]:
mt_wgs_clinvar_path = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/clinvar/splitMT/hail.mt"
)
mt_9 = hl.read_matrix_table(mt_wgs_clinvar_path)

In [ ]:
mt_9.filter_cols(mt_9.s == str(1736721))

In [ ]:
entries_v8[entries_v8.s==1736721]

In [ ]:
patient_id = "1736721"

target_locus = hl.parse_locus(
    "chr1:45331556",
    reference_genome="GRCh38"
)

result_mt = mt_9.filter_cols(mt_9.s == patient_id)

result_mt = result_mt.filter_rows(
    (result_mt.locus == target_locus) &
    (result_mt.alleles == ["C", "T"])
)

result_entries = result_mt.entries()

result_entries.show(n=100, width=200)

In [ ]:
!pip install pysam
import pysam
import pandas as pd

vat_file = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/aux/vat/vat_complete.bgz.tsv.gz"
)

tbi_file = vat_file + ".tbi"

# Read the actual first line of the compressed VAT
with pysam.BGZFile(vat_file, "r") as f:
    header_line = f.readline().decode("utf-8").rstrip("\n")

columns = header_line.lstrip("#").split("\t")

print(columns)
tbx = pysam.TabixFile(vat_file, index=tbi_file)

rows = tbx.fetch(
    "chr1",
    45331555,  # zero-based start
    45331556   # exclusive end
)

contig_i = columns.index("contig")
position_i = columns.index("position")
ref_i = columns.index("ref_allele")
alt_i = columns.index("alt_allele")

matches = []

for row in rows:
    fields = row.rstrip("\n").split("\t")

    if (
        fields[contig_i] == "chr1"
        and int(fields[position_i]) == 45331556
        and fields[ref_i] == "C"
        and fields[alt_i] == "T"
    ):
        matches.append(dict(zip(columns, fields)))

variant_df = pd.DataFrame(matches)
variant_df.clinvar_classification

In [ ]:
sample_id = str(sorted(
    samples_v8
    & samples_v9
    & samples_v8_filt
    & (samples_v8_filt - samples_v9_filt)
)[1])

target_locus = hl.locus("17", 43071077, reference_genome="GRCh37")

individual_mt = mt_9.filter_cols(mt_9.s == sample_id)

individual_mt.entries().show()
target_locus = hl.parse_locus(
    "chr1:45331556",
    reference_genome="GRCh38"
)

result = mt_9.filter_rows(mt_9.locus == target_locus)
result = result.filter_cols(result.s == sample_id)

genes_of_interest = [
    "AIP", "ALK", "APC", "ATM", "AXIN2", "BAP1", "BARD1", "BLM",
    "BMPR1A", "BRCA1", "BRCA2", "BRIP1", "CASR", "CDC73", "CDH1",
    "CDK4", "CDKN1B", "CDKN1C", "CDKN2A", "CEBPA", "CHEK2",
    "CTNNA1", "DICER1", "DIS3L2", "EGFR", "EPCAM", "FH", "FLCN",
    "GATA2", "GPC3", "GREM1", "HOXB13", "HRAS", "KIT", "MAX",
    "MC1R", "MEN1", "MET", "MITF", "MLH1", "MSH2", "MSH3", "MSH6",
    "MUTYH", "NBN", "NF1", "NF2", "NTHL1", "PALB2", "PDGFRA",
    "PHOX2B", "PMS2", "POLD1", "POLE", "POT1", "PRKAR1A", "PTCH1",
    "PTEN", "RAD50", "RAD51C", "RAD51D", "RB1", "RECQL4", "RET",
    "RUNX1", "SDHA", "SDHAF2", "SDHB", "SDHC", "SDHD", "SMAD4",
    "SMARCA4", "SMARCB1", "SMARCE1", "STK11", "SUFU", "TERC", "TERT",
    "TMEM127", "TP53", "TSC1", "TSC2", "VHL", "WRN", "WT1"
]

dataset_hl[result.s].show()

print(hl.grep(
    r"^chr1\t45331556\tC\tT\t",
    vat_path,
    max_count=1
))

# result.entries().show()

# # Keep only samples represented in dataset_hl.
# mt_sub = result.semi_join_cols(dataset_hl)

# # Keep only variants represented in filtered_vat_table.
# mt_sub = mt_sub.semi_join_rows(filtered_vat_table)

# mt_sub.describe()

# # Potentially expensive.
# # print(mt_sub.count())


# # ---------------------------------------------------------------------
# # Add patient metadata and complete VAT annotations
# # ---------------------------------------------------------------------

# mt_sub = mt_sub.annotate_cols(
#     metadata=dataset_hl[mt_sub.s]
# )

# mt_sub = mt_sub.annotate_rows(
#     annotations=filtered_vat_table[
#         mt_sub.locus,
#         mt_sub.alleles
#     ]
# )


# genes_literal = hl.literal(set(genes_of_interest))

# mt_filtered = mt_sub.filter_rows(
#     hl.is_defined(mt_sub.annotations.gene_symbol)
#     & genes_literal.contains(mt_sub.annotations.gene_symbol)
# )


# # ---------------------------------------------------------------------
# # Identify non-reference genotypes
# # ---------------------------------------------------------------------

# mt_filtered = mt_filtered.annotate_entries(
#     has_variant=(
#         hl.is_defined(mt_filtered.GT)
#         & mt_filtered.GT.is_non_ref()
#     )
# )

# mt_filtered.describe()


# # ---------------------------------------------------------------------
# # Create entries table and keep non-reference genotypes
# # ---------------------------------------------------------------------

# entries_table = mt_filtered.entries()

# entries_table = entries_table.filter(
#     entries_table.has_variant
# )

# entries_table.to_pandas()


In [ ]:
pd.set_option("display.max_columns", None)
entries_v8[entries_v8.s == int(sorted(
    samples_v8
    & samples_v9
    & samples_v8_filt
    & (samples_v8_filt - samples_v9_filt)
)[1])]

In [ ]:
meta_v9[meta_v9.s==int(sorted(
    samples_v8
    & samples_v9
    & samples_v8_filt
    & (samples_v8_filt - samples_v9_filt)
)[1])]

In [ ]:
import hail as hl
import pandas as pd


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

mt_wgs_clinvar_path = (
    "/home/jupyter/workspace/cdrv9/"
    "vwb-aou-datasets-controlled-v9/v9/wgs/short_read/"
    "snpindel/clinvar/splitMT/hail.mt"
)

metadata_output_path = (
    f"{GENETIC_FOLDER}/metadata_v9.csv"
)

entries_output_path = (
    f"{GENETIC_FOLDER}/entries_table_full_v9.csv"
)


# ---------------------------------------------------------------------
# Genes of interest
# ---------------------------------------------------------------------

# Keep your complete existing genes_of_interest list here.

genes_literal = hl.literal(set(genes_of_interest))


# ---------------------------------------------------------------------
# Read ClinVar MatrixTable
# ---------------------------------------------------------------------

mt = hl.read_matrix_table(mt_wgs_clinvar_path)

mt.describe()


# ---------------------------------------------------------------------
# Select cohort samples
# ---------------------------------------------------------------------

# dataset_hl must be keyed by s.
# This is harmless if it is already keyed correctly.
dataset_hl = dataset_hl.key_by("s")

mt_sub = mt.semi_join_cols(dataset_hl)


# ---------------------------------------------------------------------
# Keep VAT-selected variants
# ---------------------------------------------------------------------

# Use the uniquely keyed variant-level table from Cell 3.
mt_sub = mt_sub.semi_join_rows(
    filtered_vat_by_variant
)


# ---------------------------------------------------------------------
# Add patient metadata
# ---------------------------------------------------------------------

mt_sub = mt_sub.annotate_cols(
    metadata=dataset_hl[mt_sub.s]
)


# ---------------------------------------------------------------------
# Add all matching VAT transcript annotations
# ---------------------------------------------------------------------

vat_lookup = filtered_vat_by_variant[
    mt_sub.locus,
    mt_sub.alleles,
]

mt_sub = mt_sub.annotate_rows(
    annotations=vat_lookup.annotations,
    gene_symbols=vat_lookup.gene_symbols,
    transcripts=vat_lookup.transcripts,
    consequences=vat_lookup.consequences,
    clinvar_classifications=(
        vat_lookup.clinvar_classifications
    ),
)

mt_sub.describe()


# ---------------------------------------------------------------------
# Export sample metadata
# ---------------------------------------------------------------------

metadata_table = mt_sub.cols()

metadata_df = metadata_table.to_pandas()

print(metadata_df.head())
print("Metadata shape:", metadata_df.shape)

metadata_df.to_csv(
    metadata_output_path,
    index=False,
)

print(f"Saved metadata to: {metadata_output_path}")


# ---------------------------------------------------------------------
# Keep variants annotated to any gene of interest
# ---------------------------------------------------------------------

mt_filtered = mt_sub.filter_rows(
    hl.is_defined(mt_sub.gene_symbols)
    & hl.any(
        lambda gene: genes_literal.contains(gene),
        mt_sub.gene_symbols,
    )
)


# ---------------------------------------------------------------------
# Identify non-reference genotypes
# ---------------------------------------------------------------------

mt_filtered = mt_filtered.annotate_entries(
    has_variant=(
        hl.is_defined(mt_filtered.GT)
        & mt_filtered.GT.is_non_ref()
    )
)

mt_filtered.describe()


# ---------------------------------------------------------------------
# Create entries table and keep variant carriers
# ---------------------------------------------------------------------

entries_table = mt_filtered.entries()

entries_table = entries_table.filter(
    entries_table.has_variant
)

entries_table.describe()


# ---------------------------------------------------------------------
# Convert to pandas and save
# ---------------------------------------------------------------------

entries_table_df = entries_table.to_pandas()

print(entries_table_df.head())
print("Entries table shape:", entries_table_df.shape)

entries_table_df.to_csv(
    entries_output_path,
    index=False,
)

print(f"Saved entries table to: {entries_output_path}")

### Deprecated

In [ ]:
print('hi')

In [ ]:
# Load the existing filtered MatrixTable
mt = hl.read_matrix_table(f"{GENETIC_FOLDER}/filtered_nonref.mt")

# Restore all VAT annotation fields
mt = mt.annotate_rows(
    annotations=filtered_vat_table[mt.locus, mt.alleles]
)

# Restore the V8 column, if needed
mt = mt.annotate_entries(
    has_variant=hl.is_defined(mt.GT) & mt.GT.is_non_ref()
)

# Recreate only the entries table
entries_table = mt.entries()

# Convert and save
entries_table_df = entries_table.to_pandas()

entries_table_df.to_csv(
    f"{GENETIC_FOLDER}/entries_table_full_v9.csv",
    index=False
)

print(entries_table_df.shape)

In [ ]:
mt = hl.read_matrix_table(filtered_mt_path)
# Extract only needed fields
entries_table = mt.entries()

# Write distributed Hail Table first
entries_ht_path = f"{GENETIC_FOLDER}/entries_table_v9.ht"

entries_table = entries_table.checkpoint(
    entries_ht_path,
    overwrite=True
)

# Export as multiple compressed TSV shards
entries_table.export(
    f"{GENETIC_FOLDER}/entries_table_v9.tsv.bgz",
    parallel="header_per_shard"
)

In [ ]:
entries_ht_path = f"{GENETIC_FOLDER}/entries_table_v9.ht"
entries_table = hl.read_table(entries_ht_path)
entries_table_df = entries_table.to_pandas()

entries_table_df.to_csv(f'{GENETIC_FOLDER}/entries_table_full_v9.csv', index=False)

In [ ]:
entries = pd.read_csv(f"{GENETIC_FOLDER}/entries_table_full_v9.csv")

In [ ]:
entries[entries['gene_symbol']=='SDHB']

In [ ]:
mt_wgs_clinvar_path = '/home/jupyter/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/clinvar/splitMT/hail.mt'

mt = hl.read_matrix_table(mt_wgs_clinvar_path)
mt.describe()

In [ ]:
# Select only the samples in dataset_hl
mt_sub = mt.semi_join_cols(dataset_hl)
# Select only the variants in filtered_vat_table
mt_sub = mt_sub.semi_join_rows(filtered_vat_table)
mt_sub.describe()

# Annotate MatrixTable columns with metadata
mt_sub = mt_sub.annotate_cols(metadata=dataset_hl[mt_sub.s])
# Annotate MatrixTable rows with variant annotations
mt_sub = mt_sub.annotate_rows(annotations=filtered_vat_table[mt_sub.locus, mt_sub.alleles])
# Extract the column fields (metadata and sample ID 's') into a Hail Table
metadata_table = mt_sub.cols()

# Convert the Hail Table to a Pandas DataFrame
metadata_df = metadata_table.to_pandas()

# Save the DataFrame to a CSV file
metadata_df.to_csv(f'{GENETIC_FOLDER}/metadata_v9.csv', index=False)

In [ ]:
# Kind of redundant though
# Filter the MatrixTable to specific genes
genes_of_interest = ['AIP', 'ALK', 'APC', 'ATM', 'AXIN2', 'BAP1', 'BARD1', 'BLM', 'BMPR1A', 'BRCA1', 'BRCA2', 
                     'BRIP1', 'CASR', 'CDC73', 'CDH1', 'CDK4', 'CDKN1B', 'CDKN1C', 'CDKN2A', 'CEBPA', 'CHEK2', 
                     'CTNNA1', 'DICER1', 'DIS3L2', 'EGFR', 'EPCAM', 'FH', 'FLCN', 'GATA2', 'GPC3', 'GREM1', 
                     'HOXB13', 'HRAS', 'KIT', 'MAX', 'MC1R', 'MEN1', 'MET', 'MITF', 'MLH1', 'MSH2', 'MSH3', 
                     'MSH6', 'MUTYH', 'NBN', 'NF1', 'NF2', 'NTHL1', 'PALB2', 'PDGFRA', 'PHOX2B', 'PMS2', 
                     'POLD1', 'POLE', 'POT1', 'PRKAR1A', 'PTCH1', 'PTEN', 'RAD50', 'RAD51C', 'RAD51D', 'RB1', 
                     'RECQL4', 'RET', 'RUNX1', 'SDHA', 'SDHAF2', 'SDHB', 'SDHC', 'SDHD', 'SMAD4', 'SMARCA4', 
                     'SMARCB1', 'SMARCE1', 'STK11', 'SUFU', 'TERC', 'TERT', 'TMEM127', 'TP53', 'TSC1', 'TSC2', 
                     'VHL', 'WRN', 'WT1']

mt_filtered = mt_sub.filter_rows(hl.literal(genes_of_interest).contains(mt_sub.annotations.gene_symbol))
mt_filtered = mt_filtered.annotate_entries(has_variant=mt_filtered.GT.is_non_ref())
mt_filtered.describe()

In [ ]:
entries_table = mt_filtered.entries()
# Filter entries to include only non-reference variants
entries_table = entries_table.filter(entries_table.has_variant)
entries_table.describe()

# Analysis

In [ ]:
entries_v9 = pd.read_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_full_v9.csv')
entries_v8 = pd.read_csv('/home/jupyter/workspace/genetic_bucket/data/entries_table_full_v8.csv')

In [ ]:
samples_v8_filt = set(entries_v8["s"].dropna().unique())
samples_v9_filt = set(entries_v9["s"].dropna().unique())


print("v8 total:", len(samples_v8_filt))
print("v9 total:", len(samples_v9_filt))
print("v8 also in v9:", len(samples_v8_filt & samples_v9_filt))
print("v8 missing from v9:", len(samples_v8_filt - samples_v9_filt))
print("new in v9:", len(samples_v9_filt - samples_v8_filt))

In [ ]:
print(entries_v9.s.nunique())
print(entries_v8.s.nunique())

In [ ]:
entries_table_df = entries_table_df[entries_table_df['annotations.gene_symbol'] != 'MC1R']

entries_table_df = entries_table_df[
    ~(
        ((entries_table_df['annotations.gene_symbol'] == 'EPCAM') & 
         (entries_table_df['annotations.variant_type'] == 'deletion')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'PDGFRA') & 
         (entries_table_df['annotations.vid'] == '4-54281602-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'EGFR') & 
         (entries_table_df['annotations.vid'] == '7-55173126-T-C')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'RET') & 
         (entries_table_df['annotations.vid'] == '10-43100576-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'RET') & 
         (entries_table_df['annotations.vid'] == '10-43106497-G-A')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TSC1') & 
         (entries_table_df['annotations.vid'] == '9-132921940-T-G')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TMEM127') & 
         (entries_table_df['annotations.vid'] == '2-96265399-G-A')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
         (entries_table_df['annotations.vid'] == '5-1293489-C-G')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
         (entries_table_df['annotations.vid'] == '5-1268581-G-A')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'TERT') & 
         (entries_table_df['annotations.vid'] == '5-1254461-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'POLE') & 
         (entries_table_df['annotations.vid'] == '12-132680048-T-C')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'POLE') & 
         (entries_table_df['annotations.vid'] == '12-132677577-C-T')) |  
        ((entries_table_df['annotations.gene_symbol'] == 'ALK') & 
         (entries_table_df['annotations.vid'] == '2-29220747-C-T'))
    )
]



In [ ]:
entries_table_df.to_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_filt_v8.csv')